In [ ]:
import tensorflow as tf
import numpy as np

# ===== 数据加载（新版写法）=====
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype(np.float32)
x_test  = x_test.reshape(-1, 784).astype(np.float32)
y_train = tf.keras.utils.to_categorical(y_train, 10).astype(np.float32)
y_test  = tf.keras.utils.to_categorical(y_test,  10).astype(np.float32)

def next_batch(x, y, batch_size):
    idx = np.random.choice(len(x), batch_size, replace=False)
    return x[idx], y[idx]

# ===== 超参数 =====
learning_rate  = 1e-4
keep_prob_rate = 0.7
max_epoch      = 2000

# ===== 关闭TF2的即时模式，启用TF1风格的Session =====
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

def compute_accuracy(v_xs, v_ys):
    global prediction
    y_pre = sess.run(prediction, feed_dict={xs: v_xs, keep_prob: 1})
    correct_prediction = tf.equal(tf.argmax(y_pre, 1), tf.argmax(v_ys, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
    result = sess.run(accuracy, feed_dict={xs: v_xs, ys: v_ys, keep_prob: 1})
    return result

def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

xs        = tf.placeholder(tf.float32, [None, 784]) / 255.
ys        = tf.placeholder(tf.float32, [None, 10])
keep_prob = tf.placeholder(tf.float32)
x_image   = tf.reshape(xs, [-1, 28, 28, 1])

W_conv1 = weight_variable([5, 5, 1, 32])
b_conv1 = bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

W_conv2 = weight_variable([5, 5, 32, 64])
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

W_fc1       = weight_variable([7 * 7 * 64, 1024])
b_fc1       = bias_variable([1024])
h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])
h_fc1       = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop  = tf.nn.dropout(h_fc1, keep_prob)

W_fc2      = weight_variable([1024, 10])
b_fc2      = bias_variable([10])
prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)

cross_entropy = tf.reduce_mean(
    -tf.reduce_sum(ys * tf.log(prediction), reduction_indices=[1])
)
train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)

with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    for i in range(max_epoch):
        batch_xs, batch_ys = next_batch(x_train, y_train, 100)
        sess.run(train_step, feed_dict={xs: batch_xs, ys: batch_ys, keep_prob: keep_prob_rate})
        if i % 100 == 0:
            print(f"step {i}:", compute_accuracy(x_test[:1000], y_test[:1000]))


Instructions for updating:
non-resource variables are not supported in the long term

Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.

step 0: 0.085
step 100: 0.085
step 200: 0.085
step 300: 0.085
step 400: 0.085
step 500: 0.085
step 600: 0.085
step 700: 0.085
